# Document-oriented Practice Lab
## Anomaly Detection Methods Database

**Master IASD — DataAES — 2025**  
Paul Boniol

---

This notebook guides you through a MongoDB lab on a dataset of **133 anomaly detection methods**
and **13 algorithmic families** from [TSADtaxonomy](https://boniolp.github.io/TSADtaxonomy/).

| | |
|---|---|
| **Student name** | _____________________________________________ |
| **Student seed (Part 2)** | __________ |

---

> ⚠️ **The database contains inconsistencies.** Some words have inconsistent capitalization (Sup or sup), and some names are either fully written out or use only the initial (Themis Palpanas or T. Palpanas). Despite these small inconsistencies, they all refer to the same entity (e.g., Themis Palpanas and T. Palpanas are the same person, and Sup and sup are the same category).


## 0 · Setup

Run the cell below once to install pymongo (skip if already installed).


In [ ]:
!pip install pymongo --quiet


In [ ]:
from pymongo import MongoClient, ASCENDING, DESCENDING
import json, glob, os, hashlib, random, pprint

# Connect to a local MongoDB instance (default port 27017)
client = MongoClient("mongodb://localhost:27017/")
db = client["tsad_lab"]

print("Connected — MongoDB version:", client.server_info()["version"])


## Hash utility functions

For each response that output a list of documents, please print the correspodning hash.

In [2]:
def _canonical(obj):
    """Recursively strip _id and convert non-JSON types to str."""
    if isinstance(obj, dict):
        return {k: _canonical(v) for k, v in sorted(obj.items()) if k != '_id'}
    if isinstance(obj, list):
        return [_canonical(i) for i in obj]
    if isinstance(obj, (bool, int, float, str, type(None))):
        return obj
    return str(obj)   # ObjectId, datetime, Decimal128 …

def result_hash(result):
    """16-hex fingerprint, list sorted (use for unordered results)."""
    c = _canonical(result)
    if isinstance(c, list):
        c = sorted(c, key=lambda x: json.dumps(x, sort_keys=True, ensure_ascii=False))
    return hashlib.sha256(
        json.dumps(c, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()[:16]

def ordered_hash(result):
    """16-hex fingerprint, preserves list order (use for sorted/ranked results)."""
    c = _canonical(result)
    return hashlib.sha256(
        json.dumps(c, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()[:16]

print("Hash utilities ready.")


Hash utilities ready.


## 0 · Data Loading (Part 1)

Run this cell once to load the base dataset.  
Set `DATA_DIR` to the folder that contains the JSON files.


In [ ]:
DATA_DIR = "./data"   # ← adjust if needed

methods, families = [], []
for fp in sorted(glob.glob(os.path.join(DATA_DIR, '*.json'))):
    doc = json.load(open(fp, encoding='utf-8'))
    (methods if 'category' in doc else families).append(doc)

db.methods.drop()
db.families.drop()
db.methods.insert_many(methods)
db.families.insert_many(families)

print(f"Loaded {db.methods.count_documents({})} methods "
      f"and {db.families.count_documents({})} families.")


---
# Part 1 — Getting Started *(in-class)*

These questions are completed **during the lab session**.  
Write your pymongo query in the code cell below each question.


## Section A — Inserting and Exploring the Data


**Q1.** Verify the data insertion by listing **all** documents from each collection.

In [ ]:
# Your answer here. Print result_hash for each collection.


**Q2.** Count the total number of documents in the `methods` collection.

In [ ]:
# Your answer here


**Q3.** Count the total number of documents in the `families` collection.

In [ ]:
# Your answer here


**Q4.** List all **distinct category names** present in the `methods` collection.

In [ ]:
# Your answer here


**T1.** Suppose that, instead of MongoDB, you must store the same data
(the methods and families collections) in a
relational database (e.g. PostgreSQL).


**(a)**. Propose a relational schema able to store these data. For each table, give its attributes and their types, its **primary key**,
its **foreign keys**, and any relevant integrity constraints

**(b)** Justify your design. Which fields of the JSON documents
cannot be stored *as is* in a relational table, and how do you handle them?
Which normal form does your schema satisfy? How does your schema handle the
inconsistencies mentioned in Section~2?

*Your answer here*

## Section B — Filtering and Projection


**Q5.** Find all methods that handle **Multivariate** time series. Display only `name` and `category`.

In [ ]:
# Your answer here. Print result_hash of the result.


**Q6.** Find all methods published **after 2018** that support **streaming** (`Stream = True`). Show `name`, `year`, and `category` only.

In [ ]:
# Your answer here. Print result_hash of the result.


**Q7.** Find the **5 most recently published** methods. Show `name`, `year`, and `category`. Sort them first by year descending then by name ascending.

In [ ]:
# Your answer here. Print ordered_hash of the result.


**Q8.** Find all methods whose `paper` field contains the word **"neural"** (case-insensitive).

In [ ]:
# Your answer here. Print result_hash of the result.


**Q9.** Find all methods that have a code repository (the `code` field exists *and* is not an empty string).

In [ ]:
# Your answer here. Print result_hash of the result.


**Q10.** Find all methods that do **not** have a `snippet` field. Display only their names.

In [ ]:
# Your answer here. Print result_hash of the result.


**Q11.** Find methods published between **2015 and 2020 inclusive** that are **Unsupervised**. Sort by year ascending then name ascending.

In [ ]:
# Your answer here. Print ordered_hash of the result.


**Q12.** Find all methods where **Eamonn Keogh** is listed as an author.

In [ ]:
# Your answer here. Print ordered_hash of the result.


**T2**. We now imagine a much larger version of the `methods` collection
(e.g. hundreds of millions of documents describing papers), which no longer fits
on a single server and must therefore be **distributed (sharded)** over
several servers. We assume that the **most frequent queries** retrieve papers
published within a given **time range**.

**(a)** What is the most commonly used sharding strategy in MongoDB?
Describe how documents are distributed among the servers with this strategy, and
how the query above would be executed.

**(b)** Which sharding strategy would you
choose for this workload? Justify your choice by comparing it with the strategy of (a).

**(c)** What are the limits or risks of the strategy you proposed in (b),
and how could you mitigate them?

*Your answer here*

## Section C — Aggregation


**Q13.** Count the number of methods **per category**. Sort from the largest to the smallest group. In case of equality, sort by lexicographical order based on the category name.

In [3]:
# Your answer here. Print ordered_hash of the result.


**Q14.** Find all categories that contain **more than 10 methods**. Sort from the largest to the smallest group.

In [ ]:
# Your answer here. Print ordered_hash of the result.


**Q15.** Compute the **number of methods per supervision type** (`Sup` field).  

In [ ]:
# Your answer here. Print ordered_hash of the result.


**Q16.** List all **distinct author names** across all method documents. How many unique authors are there in total?

In [ ]:
# Your answer here. Print result_hash of the result.


**Q17.** For each category, compute the **average publication year**. Sort by average year descending.

In [ ]:
# Your answer here. Print ordered_hash of the result.


**Q18.** Count the number of **streaming** methods per category. Only show categories with at least one streaming method. Sort in descending order based on the number of streaming methods per category. In case of equality, sort by lexicographical order based on the category name.

In [ ]:
# Your answer here. Print ordered_hash of the result.


**T3**. We want to build a **search engine** on top of the `methods` and
`families` collections. A user types a free-text query in natural language,
for instance:

- *unsupervised streaming methods based on distance by author X.*

and the engine must return the methods sorted from the most to the least relevant.

This is an **open question**. Explain how you would compute the
**relevance score** of each method for a given query, and therefore the ranking.

*Your answer here*

## Section D — Updates and Schema Validation


**Q19.** Add a field `era` to every method document: **`"classic"`** for methods published before 2010, **`"modern"`** for 2010 or later. Verify by counting documents in each era.

In [ ]:
# Hint: use update_many (two calls, or a single pipeline update with $cond)

# Your answer here


**Q20.** Create a new collection `validated_methods` with a `$jsonSchema` validator enforcing:  
(a) `name` is required and must be a string;  
(b) `year` is required and must be an integer;  
(c) `Dim`, if present, must be `"Univariate"` or `"Multivariate"`.  
Insert one valid and one invalid document to confirm the schema works.

In [ ]:
# Hint: db.create_collection('validated_methods', validator={...})

# Your answer here


---
# Part 2 — Advanced Take-Home Assignment

## Part 2 Setup — Enter Your Seed

Before answering H1–H7, run the cell below with **your personal seed number**. The latter is computed from your last name (i.e., family name).  
This reloads the `methods` collection with two extra fields:

| Field | Type | Description |
|---|---|---|
| `rating` | int [1–5] | Simulated peer-review quality score |
| `citations` | int [50–5000] | Simulated citation count |

> ⚠️ **These values are unique to your seed.** Make sure you use your own seed number.


In [ ]:
# ── SET YOUR SEED HERE ───────────────────────────────────────────────
Your_name = None   # ← replace None with your last name (i.e, family name)
YOUR_SEED = int(hashlib.sha256(Your_name.strip().lower().encode('utf-8')).hexdigest(), 16) % 1000
print(YOUR_SEED)
# ─────────────────────────────────────────────────────────────────────


def _doc_rng(seed, name):
    key = f'{seed}::{name}'.encode()
    return random.Random(int(hashlib.sha256(key).hexdigest(), 16))

def enrich(doc, seed):
    doc = dict(doc)
    rng = _doc_rng(seed, doc['name'])
    doc['rating']    = rng.randint(1, 5)
    doc['citations'] = rng.randint(50, 5000)
    return doc

enriched = [enrich(m, YOUR_SEED) for m in methods]
db.methods.drop()
db.methods.insert_many(enriched)

# Quick sanity check
sample = db.methods.find_one({}, {'name':1,'rating':1,'citations':1,'_id':0})
print(f'Loaded {db.methods.count_documents({})} methods for seed {YOUR_SEED}')
print('Sample document:', sample)


## Part 2 Questions


### H1 — Category Performance Summary

For each category, compute **(a)** the **average rating** (rounded to 2 decimal places) and **(b)** the **total number of citations**.  
Return only categories with **at least 5 methods**, sorted by average rating descending, then total citations descending.


In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H2 — Prolific Cross-Category Authors

Find all authors who have co-authored papers in **at least 3 different categories**.  
For each such author, return their name and the **sorted list of distinct categories** they published in.  
Sort by number of categories descending, then by author name alphabetically.


In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H3 — Highest-Rated Category

Find the category where the **proportion of methods with rating ≥ 4** to the total number of methods is the highest.  
Tie-break: highest **total citations**. Return the category name, the proportion (rounded to 3 decimal places), and the total citations.


In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H4 — Top Cited Methods in Recent High-Performing Categories

Consider only methods published **between 2015 and 2020 inclusive**.  
Among these, find the **top-3 categories** by average citations.  
For each of these 3 categories, return the **single method with the highest individual citation count** (name, category, citations, year).  



In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H5 — Family–Method Cross-Collection Join

For each document in the **`families`** collection whose `name` matches a `category` in **`methods`**, return:  
(a) the family name, (b) the number of methods in that category,  
(c) the average rating (rounded to 2 decimal places),  
(d) whether any method in that category has a `snippet` field.  
Sort by average rating descending.  



In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H6 — Most Productive Year

Find the **single year** in which the largest number of **distinct authors** published at least one method.  
Return that year, the count of distinct authors, and the **list of author names** (in lexicographical order).


In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H7 — Supervision-Type Deep Dive

For each **supervision type** (`Sup` field), produce a summary with:  
(a) the supervision type, (b) total methods, (c) the **most common category**, (d) the average publication year (rounded to 1 decimal place),  
(e) the **names of the top-2 most cited methods**.  

Sort the results by total methods, descending.


In [ ]:

# Your answer here. Print ordered_hash of the result.


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


### H8 — Co-authorship Shortest Path

Two authors are **co-authors** if they appear together in at least one method's `authors` list. This defines a graph where authors are nodes and a co-authorship link is an edge.

**(a)** Build a `co_authors` collection in MongoDB (one document per author, with a `co_authors` list of their collaborators).  
Then write a `$graphLookup` aggregation that tries to find the **shortest path** between two specific authors — including all intermediate nodes.  
 

**(b)** Implement a Python function `find_path(start, end)` that returns the **shortest path** as a list of author names.  
Test it on the following pairs:

| Pair | Author A | Author B |
|---|---|---|
| 1 | `'Eamonn Keogh'` | `'John Paparrizos'` |
| 2 | `'Eamonn Keogh'` | `'D. Papadopoulos'` |
| 3 | `'Pavel Senin'` | `'D. Papadopoulos'` |

**(c)** Discuss the feasibility, the advantages and the limits of both (a) and (b). Can you imagine an alternative to (a) and (b)?


#### (a) Build `co_authors` and attempt `$graphLookup`


In [ ]:
# Step 1: build co-authorship adjacency dict from the methods collection
from collections import deque

adjacency = {}
# Your code here

# Step 2: insert into a co_authors collection
db.co_authors.drop()
# Your code here

# Step 3: try $graphLookup to retrieve the path
# (choose any two authors for the demo)
start_author = 'Eamonn Keogh'
end_author   = '???'   # ← pick an end author

gl_result = list(db.co_authors.aggregate([
    # Your $graphLookup pipeline here
]))
print(gl_result)


#### (b) Python — shortest co-authorship path


In [ ]:
def find_path(start, end):
    """Return the shortest co-authorship path as a list of author names, or None."""
    # Your implementation here
    pass

# Test on the required pairs
all_sorted = sorted(adjacency.keys())

test_pairs = [
    ('Eamonn Keogh', 'John Paparrizos'),   
    ('Eamonn Keogh', 'D. Papadopoulos'),  
    ('Pavel Senin',  'D. Papadopoulos'),  
]

for a, b in test_pairs:
    path, err = find_path(a, b)
    if path:
        print(f'[{len(path)-1} hop(s)]  {" → ".join(path)}')
    else:
        print(f'  {a!r} ↔ {b!r}: {err}')


> ✏️ **Explain your approach** — after running your query above, briefly describe the steps of your pipeline in plain language.
> What stages did you use and why? What traps or edge cases did you watch out for?


*Your explanation here…*


#### (c) Discuss the feasibility, the advantages and the limits of both (a) and (b). Can you imagine an alternative to (a) and (b)?


*Your explanation here…*


### H9 — Academic Descendants (Design Question)

Consider the following query:

> *"Which author has the most descendants that published at least one paper in the database?"*

Here, **descendant** means an academic descendant in the sense of the [Math Genealogy Project](https://genealogy.math.ndsu.nodak.edu/): a PhD student advised by the author, or a student of a student, etc.

**1. Can you answer this query directly with the current database?** Explain your answer.

**2. Propose a high-level solution.** Describe, in plain language or pseudo-code, your solution (building the advisor–student collection and querying it).


*Write your answer here (plain text / pseudo-code)…*
